# Notes for TinyNeRF

## 相机参考系如何变换到世界参考系

我们首先定义两个参考系与一个引理：

1.相机参考系

2.世界参考系

引理：方向向量是起始点在原点的向量，并非自由矢量。这样做的目的是为了让**方向向量只表示方向而不携带任何位置信息**，我们将所有方向向量的起始点都固定在原点就是为了删除掉位置信息，所有方向向量平等地只指示方向

相机参考系内的方向向量不受相机位置与姿态的影响，也就是当我们对相机的位矢(世界系)进行矩阵变换时，相机参考系内的向量不会做任何改变

相机的位置矢量和姿态可以在世界参考系内做任意的位移和旋转变换，相机的默认位置在原点，方向沿着$Z$轴负方向。

但我们有时必须要将相机参考系内的方向向量变换到变换到世界坐标系中，因为相机参考系下的方向向量不受相机位置与姿态的影响，但在世界系中这是不合理的，因为他无法表示“真实的”方向。


### c2w矩阵

对于相机，我们通常用三个向量表示相机姿态，一个向量表示相机位矢:

<img src="./notes/1.jpg" width=40%>

$\vec{z}_{c}$ , $\vec{x}_{c}$ , $\vec{y}_{c}$ 用于描述相机的姿态，他们都是单位方向向量，遵循上方的引理

$\vec{r}_{c}$ 用于描述相机的位置

对于相机参考系中的任一方向向量 $\vec{d}_{c}=[a,b,c]$,它真实的表示应该是：
$$
\vec{d}_{c}=a \cdot \vec{x}_{c} + b \cdot \vec{y}_{c} + c\cdot \vec{z}_{c}
$$
也就是说方向向量 $\vec{d}_{c}$ 是以 ($\vec{x}_{c}$ , $\vec{y}_{c}$ , $\vec{z}_{c}$)为基底表示的

但有一个很有意思地方就是，我们在相机参考系下观察这个向量，它指向的方向和世界参考系中的:
$$
\vec{d}_{w}=a \cdot \vec{i} + b \cdot \vec{j} + c\cdot \vec{k}
$$
指向的方向一致

所谓的方向向量变换参考系，本质上就是：

将方向向量从由相机参考系的基底 ($\vec{x}_{c}$ , $\vec{y}_{c}$ , $\vec{z}_{c}$) 表示变为由世界参考系的基底 ($\vec{i}$ , $\vec{j}$ , $\vec{k}$) 表示

我们目前已知的是
$$
\vec{d}_{c}=a \cdot \vec{x}_{c} + b \cdot \vec{y}_{c} + c\cdot \vec{z}_{c}
$$
我们可以这样做，将 ($\vec{x}_{c}$ , $\vec{y}_{c}$ , $\vec{z}_{c}$) 三个基向量用 ($\vec{i}$ , $\vec{j}$ , $\vec{k}$) 表示即可

例如：

在相机参考系中
$$
\vec{x}_{c}=(1,\ 0,\ 0),\quad \vec{y}_{c}=(0,\ 1,\ 0),\quad \vec{z}_{c}=(0,\ 0,\ 1)
$$
因为基底是 ($\vec{x}_{c}$ , $\vec{y}_{c}$ , $\vec{z}_{c}$)

在世界参考系中(这只是一个例子)：
$$
\vec{x}_{c}=\left(\frac{1}{\sqrt2},\ \frac{1}{\sqrt2},\ 0\right),\quad
\vec{y}_{c}=\left(-\frac{1}{\sqrt2},\ \frac{1}{\sqrt2},\ 0\right),\quad
\vec{z}_{c}=(0,\ 0,\ 1)
$$
此时的基底是 ($\vec{i}$ , $\vec{j}$ , $\vec{k}$)

之后将三个式子带入：
$$
\vec{d}_{c}=a \cdot \vec{x}_{c} + b \cdot \vec{y}_{c} + c\cdot \vec{z}_{c}
$$
即可得到由 ($\vec{i}$ , $\vec{j}$ , $\vec{k}$) 表示的 $\vec{d}_{c}$

我们这样来理解这个过程：
$$
\vec{d}_{c}=a \cdot \vec{x}_{c} + b \cdot \vec{y}_{c} + c\cdot \vec{z}_{c}
$$
表示 $\vec{d}_{c}$ 沿着 $\vec{x}_{c}$ 方向有 $a$ 份，沿着 $\vec{y}_{c}$ 方向有 $b$ 份，沿着 $\vec{z}_{c}$ 方向有 $c$ 份，我们在变换中想传递给新坐标系的信息是这个向量在原坐标系中的份数

以上过程用矩阵表示就是：
$$

[a,b,c] \times

\begin{bmatrix}
| & | & |  \\
x_{c-w} & y_{c-w} & z_{c-w}  \\
| & | & | 
\end{bmatrix}

$$

## 相机方向向量的坐标

<img src="./notes/2.jpg" width=70%>

(上图展示OpenCV以及OpenGL的相机默认坐标轴的方向，我们在此使用OpenGL的默认坐标轴方向)

在真实的物理世界中，一个像素的局部坐标其实是 $(X_{pixel}, Y_{pixel}, -\text{focal})$。

为了简化计算，工程上通常会对这个向量进行缩放：把整个向量除以 focal,即$(\frac{X_{pixel}}{\text{focal}}, \frac{Y_{pixel}}{\text{focal}}, -1)$。

这样一来，物理上的成像平面就被按比例“拉近”到了距离光心正好为 1 的位置。此时的 X 和 Y 坐标（即 (i-W*0.5)/focal），就变成了由相似三角形算出来的斜率（正切值）。这使得后续的计算完全脱离了具体的像素单位，变成了纯粹的空间几何关系

对于由光心($Origin$)经成像平面发出的射线，我们通常使用直线的参数方程来描述直线上的点：
$$
\vec{r} = \vec{o} + t \vec{d}
$$
其中 $t$ 表示沿此射线单次采样的步长

采样率 (Ratio of Sampling) 一般用单位距离上的采样次数表示